# 01_target_definition
- 노트북의 목표
   - cut-off 기반 이탈/유지 라벨 직접 설계
   - 데이터 누수 없는 피처 생성 (cut-off 이전 데이터만 사용)
   - 단계적 피처 고도화
   - 분석용 데이터셋 도출 (hnm_churn_dataset_v3.csv)
- v2 대비 변경 사항
   - 마지막 5번 구매 간격을 개별 컬럼으로 펼친 피처 추가(시퀀스 정보를 트리 모델이 읽을 수 있는 형태로 변환)
   - 목적: LSTM/RNN 없이도 시퀀스 정보 자체가 성능 개선에 기여하는지 검증

## 목차
1. 데이터 로드
   - 데이터 출처 : https://www.kaggle.com/datasets/sohyunjun0401/h-and-m-personalized-fashion-data
   - 데이터 라벨링 목적 : H&M 거래 데이터를 활용하여 고객 이탈 여부를 예측하기 위한 마스터 데이터셋 생성
2. 고객별 최대 구매 간격 계산 
   - 고객 별로 '재구매하는 데 걸린 시간'을 전부 측정
3. 이탈 기준일(Cut-off) 도출
   - 통계치(75분위수)를 확인해보니 203일 도출
   - 데이터 종료일에서 203일을 뺀 '2020-03-03'을 Cut-off Date로 확정
4. 라벨(Churn) 생성
   - Cut-off 이전 가입 유저 중 '30일 이상 활동한 유저'만 필터링
   - Cut-off 이후(미래 203일)에 구매 기록이 없으면 churn=1(이탈) 부여 => 결과물 : df_label (customer_id, churn)
5. 파생 feature 생성 (ver2와 동일 라벨 생성, 기존 22개 피처)
   - 5-1. 기본 RFM : 총 구매 횟수(frequency), 구매 금액 합계/평균/최대값
   - 5-2. Recency : Cut-off 일자 기준으로 마지막 구매일이 며칠 전인가?
   - 5-3. 활동성 : 최근 90일 / 180일 동안 각각 몇 번 구매했는가?
   - 5-4. 구매 간격 : 유저별 평균 구매 주기 및 주기 편차(std)
   - 5-5. 구매 다양성 : 얼마나 다양한 상품과 카테고리를 샀는가?
   - 5-6. 구매 취향 : 가장 자주 산 카테고리/인덱스 그룹(패션 스타일)이 무엇인가? 
   - 5-7. 구매 트렌드 : 구매 빈도와 금액이 시간이 갈수록 줄어드는지를 측정
   - 5-8. 구매 다양성 변화 : 최근 90일 구매 고유 상품 수. 낮을수록 쇼핑 스펙트럼이 줄어듦
   - 5-9. 구매 리듬 : 최근들어 구매가 뜸해졌는지
6. 마지막 5회 구매 간격 feature 생성
   - 마지막 5회 구매 간격을 개별 컬럼으로 모델에 전달
7. 데이터 결합
   - 라벨을 중심에 두고, 각 파생변수들과 고객 메타정보를 customer_id 기준으로 Left Join
8. 데이터 누수 검증
   - 중복 데이터 확인
   - 데이터 누수 확인
9. 데이터 저장
10. 최종 데이터셋 정의서

### 1. 데이터 로드

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

#시각화 한글폰트
plt.rc('font', family='Malgun Gothic')
plt.rcParams['axes.unicode_minus'] = False
sns.set_theme(style='whitegrid')

trans     = pd.read_csv('../data/raw/transactions_train.csv', nrows=100000)
customers = pd.read_csv('../data/raw/customers.csv')
articles  = pd.read_csv('../data/raw/articles.csv')

print(f'Transactions : {trans.shape}')
print(f'Customers    : {customers.shape}')
print(f'Articles     : {articles.shape}')


Transactions : (100000, 5)
Customers    : (1371980, 7)
Articles     : (105542, 25)


In [3]:
trans.head()

,t_dat,customer_id,article_id,price,sales_channel_id
0,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,663713001,0.050831,2
1,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,541518023,0.030492,2
2,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,505221004,0.015237,2
3,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,685687003,0.016932,2
4,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,685687004,0.016932,2


In [4]:
articles.head()

,article_id,product_code,prod_name,product_type_no,product_type_name,product_group_name,graphical_appearance_no,graphical_appearance_name,colour_group_code,colour_group_name,...,department_name,index_code,index_name,index_group_no,index_group_name,section_no,section_name,garment_group_no,garment_group_name,detail_desc
0,108775015,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,9,Black,...,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
1,108775044,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,10,White,...,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
2,108775051,108775,Strap top (1),253,Vest top,Garment Upper body,1010017,Stripe,11,Off White,...,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
3,110065001,110065,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,9,Black,...,Clean Lingerie,B,Lingeries/Tights,1,Ladieswear,61,Womens Lingerie,1017,"Under-, Nightwear","Microfibre T-shirt bra with underwired, moulde..."
4,110065002,110065,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,10,White,...,Clean Lingerie,B,Lingeries/Tights,1,Ladieswear,61,Womens Lingerie,1017,"Under-, Nightwear","Microfibre T-shirt bra with underwired, moulde..."


### 2. 고객별 최대 구매 간격 계산

In [5]:
# 전체 거래 데이터 로드
trans_all = pd.read_csv('../data/raw/transactions_train.csv', usecols=['t_dat', 'customer_id'])
trans_all['t_dat'] = pd.to_datetime(trans_all['t_dat'])

# 유저별 구매 간격 계산
trans_sorted = trans_all.sort_values(['customer_id', 't_dat'])
trans_sorted['diff'] = trans_sorted.groupby('customer_id')['t_dat'].diff().dt.days
user_max_diff = trans_sorted.groupby('customer_id')['diff'].max().dropna()
print(f'구매 간격 계산 완료: {len(user_max_diff):,}명')

구매 간격 계산 완료: 1,230,767명


### 3. 이탈 기준일(Cut-off) 도출
- 데이터에서 고객들의 평균적인 재구매 간격을 확인
- 구매 횟수가 1회인 고객은 직전 구매일이 없어서 diff값이 NaN

In [ ]:
# 기초 통계량 확인
median_diff = user_max_diff.median()
mean_diff = user_max_diff.mean()
q75_days_calc = user_max_diff.quantile(0.75)

print("▶ 이탈 기준일 설정:")
print(f" - 중앙값(50%) : {median_diff}일")
print(f" - 평균값(Mean): {mean_diff:.1f}일")
print(f" - 75분위수(Q3): {q75_days_calc}일")

# 전후 구간 밀도 분석 함수 (꺾임점 증명용)
def analyze_window_density(data, standard_days, window=30):
    before_count = ((data >= standard_days - window) & (data < standard_days)).sum() # 기준일 전 후 30일 구간 최대 구매 공백기 계산
    after_count = ((data >= standard_days) & (data < standard_days + window)).sum()
    total = len(data)
    before_pct = (before_count / total) * 100
    after_pct = (after_count / total) * 100
    drop_rate = ((before_pct - after_pct) / before_pct) * 100 if before_pct > 0 else 0 # 급감률 계산
    print(f" 기준일 [{int(standard_days)}일] 전후 {window}일 밀도 급감률: {drop_rate:.1f}%")

print("▶ 주요 후보 날짜별 데이터 감속 비율 비교:")
analyze_window_density(user_max_diff, standard_days=median_diff)    # 중앙값 기준
analyze_window_density(user_max_diff, standard_days=mean_diff)      # 평균값 기준
analyze_window_density(user_max_diff, standard_days=q75_days_calc)  # 75분위수 기준

- 해당 날짜가 진짜 고객 행동이 바뀌는 시점일까? 검증
    - 75분위수인 203일의 전후 30일 추세 파악 : 21.3% 구매 감소하여 203일을 cut-off 기준일로 확정 

In [ ]:
# 고객별 최대 구매 간격의 장기 분포와 cut-off 기준일 선정 근거
plt.figure(figsize=(12, 5))
sns.histplot(user_max_diff, bins=60, kde=True, color='skyblue', stat='percent')
plt.axvline(median_diff, color='green', linestyle='--', label=f'Median ({median_diff}일)')
plt.axvline(mean_diff, color='red', linestyle='--', label=f'Mean ({mean_diff:.1f}일)')
plt.axvline(q75_days_calc, color='purple', linestyle='-', linewidth=2.5, label=f'75th Percentile ({int(q75_days_calc)}일)')
plt.title("고객별 최대 구매 간격의 장기 분포와 cut-off 기준일 선정 근거", fontsize=13, fontweight='bold')
plt.xlabel("Maximum Days Between Purchases")
plt.ylabel("Percentage of Users (%)")
plt.xlim(-20, 600)
plt.legend()
plt.show()

In [ ]:
# 검증한 통계치를 기반으로 '75분위수'를 최종 이탈 기준일로 확정
q75_days = int(q75_days_calc)

# 데이터 시작일, 종료일 및 cut_off_date 계산
min_date = trans_all['t_dat'].min()
max_date = trans_all['t_dat'].max()
cut_off_date = max_date - pd.Timedelta(days=q75_days)

print("[전체 거래 데이터 기간 확인]")
print(f"-   최종 이탈 기준일  : {q75_days}일 (Q3)")
print(f"-   데이터 시작일 (Min Date) : {min_date.date()}")
print(f"-   데이터 종료일 (Max Date)    : {max_date.date()}")
print(f"-   cut_off 기준일    : {cut_off_date.date()}")
print(f"-   전체 데이터 수집 기간   : {(max_date - min_date).days + 1}일")

print("[최종 시간 축 구조]")
print(f"    - 데이터 시작일부터 cut_off_date까지 : {trans_all['t_dat'].min().date()} ~ {cut_off_date.date()} (약 {len(trans_all[trans_all['t_dat'] <= cut_off_date])}건)")
print(f"    - cut_off_date부터 데이터 종료일까지 : {cut_off_date.date()} ~ {max_date.date()} (약 {len(trans_all[trans_all['t_dat'] > cut_off_date])}건)")

### 4. 라벨(Churn) 생성
- cut_off 이후에 구매했으면 → 유지 (0)
- cut_off 이후에 구매 안 했으면 → 이탈 (1)
- 가입 30일 미만 초기 유저는 제외 (판단 근거가 부족)
- 데이터 누수 방지를 위해 라벨 생성에 쓴 임시 컬럼(`recency_for_label`, `threshold`)은 바로 삭제

In [ ]:
# 1. 시점 격리: 피처용 과거 데이터와 라벨용 미래 데이터로 명확히 분리
df_past_base = trans_all[trans_all['t_dat'] <= cut_off_date]
df_future_base = trans_all[trans_all['t_dat'] > cut_off_date]

# 고객의 첫 구매일과 가입기간(tenure) 계산 (cut_off 이전 데이터로만 계산)
user_stats = df_past_base.groupby('customer_id')['t_dat'].agg(['min', 'max']).reset_index()
user_stats.columns = ['customer_id', 'first_buy', 'last_buy_past']
user_stats['tenure'] = (cut_off_date - user_stats['first_buy']).dt.days

# 3. 30일 미만 초기 고객 제외 (30일 이상 활동 고객만 분석 대상으로 선정)
df_label = user_stats[user_stats['tenure'] >= 30].copy()

# 4. 라벨 생성
# 미래 구간(203일 동안)에 단 한 번이라도 구매한 이력이 있는 유니크한 고객 ID만 추출
active_users_in_future = df_future_base['customer_id'].unique()

# 미래에 구매 기록이 있다면 -> Churn = 0 (유지)
# 미래에 구매 기록이 없다면 -> Churn = 1 (이탈)
df_label['churn'] = df_label['customer_id'].isin(active_users_in_future).apply(lambda x: 0 if x else 1)

# customer_id와 churn만 남기고 정리
df_label = df_label[['customer_id', 'churn']]

print(f"1. 분석 대상 유저 수 (가입 30일 이상): {len(df_label):,}명")
print(f"\n2. churn 라벨 분포")
print(df_label['churn'].value_counts())
print(f"\n3. 이탈률: {df_label['churn'].mean() * 100:.2f}%")

### 5. 파생 feature 생성
- 모두 cut_off 이전 데이터만 사용
- 전체 거래 데이터를 로드하고, cut_off 이전으로 필터링한 뒤 모든 피처를 한번에 계산

In [ ]:
# 전체 거래 데이터 로드
trans_feat = pd.read_csv('../data/raw/transactions_train.csv',
                         usecols=['t_dat','customer_id','article_id','price'])
trans_feat['t_dat'] = pd.to_datetime(trans_feat['t_dat'])
trans_feat = trans_feat[trans_feat['t_dat'] <= cut_off_date].copy()
print(f'피처 생성 대상 거래 수: {len(trans_feat):,}건')

피처 생성 대상 거래 수: 22,991,230건


### 5-1. 기본 RFM 피처
- Recency (최근성): 고객이 얼마나 최근에 우리 가게에 왔는가?
- Frequency (빈도): 얼마나 자주 샀는가?
- Monetary (금액): 돈을 얼마나 많이 썼는가?
=> 구매 기록 자체를 집계

### 5-2. Recency 피처
- (max_date - last_buy)로 계산하면 정답 라벨과 100% 동일한 의미라 데이터 누수가 된다.
- (cut_off_date - last_buy)로 생성 : 관측 시점 기준 마지막 구매가 얼마나 오래되었나. (정답과 무관한 피처) => "마지막 구매가 xx일 전이다"

### 5-3. 활동성 피처
- 최근 90일, 180일 동안 각각 몇 번 구매했는가?
- 고객이 이탈할 때는 방문이 갑자기 뚝 끊기기보다 방문 빈도가 점점 뜸해지는 전조증상을 보임
- 모델이 고객의 최근 활동성 변화를 반영할 수 있도록 파생변수 추가

### 5-4. 구매 간격 피처
- 얼마나 규칙적으로 오는지
- 고객별 평균 구매 주기 및 주기 편차(std)

### 5-5. 구매 다양성 피처
- 얼마나 다양한 상품과 카테고리를 구매했는지 (쇼핑 스펙트럼의 넓이)

### 5-6. 구매 취향 피처
- 가장 자주 산 카테고리/인덱스 그룹(패션 스타일)이 무엇인가?

In [ ]:
# 1. RFM
user_rfm = trans_feat.groupby('customer_id').agg(
    frequency  = ('article_id','count'),
    price_mean = ('price','mean'),
    price_sum  = ('price','sum'),
    price_max  = ('price','max'),
).reset_index()

# 2. Recency
user_last_buy = trans_feat.groupby('customer_id')['t_dat'].max().reset_index()
user_last_buy.columns = ['customer_id','last_buy_before_cutoff']
user_last_buy['recency'] = (cut_off_date - user_last_buy['last_buy_before_cutoff']).dt.days
user_last_buy = user_last_buy.drop(columns=['last_buy_before_cutoff'])

# 3. 활동성
# 최근 90일, 180일 기준일
date_90d  = cut_off_date - pd.Timedelta(days=90)
date_180d = cut_off_date - pd.Timedelta(days=180)

# 최근 90일의 구매 횟수
freq_90d = (trans_feat[trans_feat['t_dat'] >= date_90d].groupby('customer_id')['article_id'].count()
            .reset_index().rename(columns={'article_id':'frequency_last_90d'}))

# 최근 180일의 구매 횟수
freq_180d = (trans_feat[trans_feat['t_dat'] >= date_180d].groupby('customer_id')['article_id'].count()
             .reset_index().rename(columns={'article_id':'frequency_last_180d'}))


# 4. 구매 간격
trans_sorted_feat = trans_feat.sort_values(['customer_id','t_dat'])
trans_sorted_feat['interval'] = trans_sorted_feat.groupby('customer_id')['t_dat'].diff().dt.days

user_interval = trans_sorted_feat.groupby('customer_id')['interval'].agg(
    avg_purchase_interval = 'mean',  # 평균 구매 간격 (짧을수록 자주 옴)
    std_purchase_interval = 'std',   # 구매 간격 편차 (작을수록 규칙적)
).reset_index()

# 5. 구매 다양성
# articles.csv에서 product_group 상품 대분류 이름 불러와 어떤 카테고리 상품인지 붙이기
trans_with_cat = pd.merge(trans_feat[['customer_id','article_id']],
                           articles[['article_id','product_group_name']],
                           on='article_id', how='left')
user_diversity = trans_with_cat.groupby('customer_id').agg(
    unique_articles       = ('article_id','nunique'),         # 구매한 고유 상품 수
    unique_product_groups = ('product_group_name','nunique'), # 구매한 고유 카테고리 수
).reset_index()

# 6. 구매 취향
trans_style = pd.merge(trans_feat[['customer_id','article_id']],
                        articles[['article_id','index_group_name','product_group_name']],
                        on='article_id', how='left')

# 가장 많이 산 것 하나만 고르는 함수
def get_most_frequent(x):
    if x.empty: return 'NONE'
    return x.value_counts().idxmax()

user_style = trans_style.groupby('customer_id').agg(
    index_group_name   = ('index_group_name',   get_most_frequent),
    product_group_name = ('product_group_name', get_most_frequent),
).reset_index()

기존 피처(RFM/Recency/활동성/간격/다양성/취향) 생성 완료


### 5-7. 구매 트렌드 피처
- 구매 빈도와 금액이 시간이 갈수록 줄어드는지를 측정
- cut_off 이전 데이터만 사용. 누수 방지.
    - `freq_decay_rate` : 최근 90일 구매 / 최근 180일 구매 → 0.5면 후반에 구매가 절반으로 줄었다는 뜻
    - `recent_price_ratio` : 최근 90일 평균 구매액 / 전체 평균 구매액 → 1보다 작으면 지출이 줄어드는 중

### 5-8. 구매 다양성 변화 피처
- `unique_articles_last_90d` : 최근 90일 구매 고유 상품 수
- `diversity_decay_rate` : 최근 90일 고유 상품 수 / 전체 고유 상품 수 → 낮을수록 쇼핑 스펙트럼이 줄어드는 중

### 5-9. 구매 리듬 피처
- `last_interval` : 마지막 구매 간격 (가장 최근 공백일 수)
- `interval_acceleration` : 마지막 구매 간격 / 평균 구매 간격 → 1보다 크면 최근 들어 구매가 뜸해진 것
- 구매가 1회뿐인 고객은 간격을 계산할 수 없으므로 NaN 유지

In [ ]:
# 7. 구매 트렌드

# 최근 90일 평균 구매액
price_90d = (trans_feat[trans_feat['t_dat'] >= date_90d].groupby('customer_id')['price'].mean()
             .reset_index().rename(columns={'price':'price_mean_last_90d'}))

# 구매 트렌드 피처 결합
user_trend = freq_90d.copy()
user_trend = pd.merge(user_trend, freq_180d, on='customer_id', how='outer')
user_trend = pd.merge(user_trend, price_90d, on='customer_id', how='left')
user_trend = pd.merge(user_trend, user_rfm[['customer_id','price_mean']], on='customer_id', how='left')

# 결측치 0으로 채우기 (해당 기간 구매 없는 고객)
user_trend['frequency_last_90d']  = user_trend['frequency_last_90d'].fillna(0)
user_trend['frequency_last_180d'] = user_trend['frequency_last_180d'].fillna(0)

# freq_decay_rate: 90일 구매 / 180일 구매
# 180일 구매가 0인 고객은 NaN 처리 (구매 이력 없음)
user_trend['freq_decay_rate'] = np.where(user_trend['frequency_last_180d'] > 0,
    user_trend['frequency_last_90d'] / user_trend['frequency_last_180d'], np.nan)

# recent_price_ratio: 최근 90일 평균 구매액 / 전체 평균 구매액
user_trend['recent_price_ratio'] = np.where(user_trend['price_mean'] > 0,
    user_trend['price_mean_last_90d'] / user_trend['price_mean'], np.nan)

user_trend = user_trend[['customer_id','freq_decay_rate','recent_price_ratio']]



# 8. 다양성 변화

# 최근 90일 고유 상품 수
diversity_90d = (trans_feat[trans_feat['t_dat'] >= date_90d].groupby('customer_id')['article_id'].nunique()
                  .reset_index().rename(columns={'article_id':'unique_articles_last_90d'}))
user_diversity_change = pd.merge(diversity_90d, user_diversity[['customer_id','unique_articles']],
                                  on='customer_id', how='right')
user_diversity_change['unique_articles_last_90d'] = user_diversity_change['unique_articles_last_90d'].fillna(0)
user_diversity_change['diversity_decay_rate'] = np.where(user_diversity_change['unique_articles'] > 0,
    user_diversity_change['unique_articles_last_90d'] / user_diversity_change['unique_articles'], np.nan)

# diversity_decay_rate
user_diversity_change = user_diversity_change[['customer_id','unique_articles_last_90d','diversity_decay_rate']]


# 9. 구매 리듬

# 고객별 마지막 구매 간격 추출
last_interval_df = (trans_sorted_feat.groupby('customer_id')['interval'].last() # 시간순 정렬된 상태에서 마지막 간격
                     .reset_index().rename(columns={'interval':'last_interval'}))

user_rhythm = pd.merge(last_interval_df, user_interval[['customer_id','avg_purchase_interval']],
                        on='customer_id', how='left')

# interval_acceleration: 마지막 간격 / 평균 간격
user_rhythm['interval_acceleration'] = np.where(user_rhythm['avg_purchase_interval'] > 0,
    user_rhythm['last_interval'] / user_rhythm['avg_purchase_interval'], np.nan)

user_rhythm = user_rhythm[['customer_id','last_interval','interval_acceleration']]

v2 신규 피처(트렌드/다양성변화/리듬) 생성 완료


### 6. 마지막 5회 구매 간격 feature 생성
- 기존 interval_acceleration은 '마지막 간격 / 평균 간격'
- 마지막 5번의 구매 간격을 개별 컬럼으로 그대로 펼쳐서 모델에 전달하려는 의도
- 구매 횟수가 5회 미만인 고객은 NaN

In [11]:
N = 5  # 펼칠 구매 간격 개수

# 고객별로 시간순 정렬된 간격을 리스트로 모음
interval_lists = (
    trans_sorted_feat.dropna(subset=['interval'])
    .groupby('customer_id')['interval']
    .apply(list)
)

print(f'구매 간격 보유 고객 수: {len(interval_lists):,}명')
print(f'평균 보유 간격 개수: {interval_lists.apply(len).mean():.1f}개')


구매 간격 보유 고객 수: 1,050,786명
평균 보유 간격 개수: 20.8개


In [12]:
# 마지막 N개를 최신순(t1=가장 최근)으로 컬럼화
def get_last_n_intervals(interval_list, n=N):
    last_n = interval_list[-n:]      # 끝에서 n개 (시간순 정렬되어 있으므로 가장 최근 n개)
    last_n_reversed = last_n[::-1]   # 최신이 먼저 오도록 뒤집기 (t1=최신)
    padded = last_n_reversed + [np.nan] * (n - len(last_n_reversed))  # 부족하면 NaN 패딩
    return padded

interval_matrix = interval_lists.apply(get_last_n_intervals)
interval_df = pd.DataFrame(
    interval_matrix.tolist(),
    index=interval_matrix.index,
    columns=[f'interval_t{i+1}' for i in range(N)]
).reset_index()

print(f'시퀀스 피처 생성 완료: {interval_df.shape}')
interval_df.head()


시퀀스 피처 생성 완료: (1050786, 6)


,customer_id,interval_t1,interval_t2,interval_t3,interval_t4,interval_t5
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,0.0,0.0,6.0,55.0,0.0
1,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,36.0,0.0,0.0,0.0,0.0
2,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,0.0,0.0,339.0,158.0,4.0
3,00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2...,0.0,NaN,NaN,NaN,NaN
4,00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...,8.0,208.0,0.0,146.0,0.0


In [ ]:
# 결측치 비율 확인 (구매 횟수가 적을수록 뒤 컬럼이 비게 됨 : 정상)
print('--- interval_t1~t5 결측치 비율(%) ---')
for col in [f'interval_t{i+1}' for i in range(N)]:
    null_pct = interval_df[col].isnull().mean() * 100
    print(f'{col} : {null_pct:.1f}%')

--- interval_t1~t5 결측치 비율(%) ---
interval_t1 : 0.0%
interval_t2 : 10.8%
interval_t3 : 18.9%
interval_t4 : 25.9%
interval_t5 : 31.4%


### 7. 데이터 결합
- 라벨을 중심에 두고, 각 파생변수들과 고객 메타정보를 customer_id 기준으로 Left Join

In [ ]:
# 고객 메타 데이터
customers_features = customers[[
    'customer_id','FN','Active','club_member_status','fashion_news_frequency','age'
]]

# merge
df_analysis = df_label.copy()

# 모든 컬럼 병합
for feat_df, name in [
    (user_rfm,              'RFM'),
    (user_last_buy,         'Recency'),
    (freq_90d,              '활동성 90일'),
    (freq_180d,             '활동성 180일'),
    (user_interval,         '구매 간격'),
    (user_diversity,        '다양성'),
    (user_style,            '취향'),
    (user_trend,            '구매 트렌드'),
    (user_diversity_change, '다양성 변화'),
    (user_rhythm,           '구매 리듬'),
    (interval_df,           '시퀀스 간격 t1~t5'),
    (customers_features,    '고객 메타'),
]:
    df_analysis = pd.merge(df_analysis, feat_df, on='customer_id', how='left')
    print(f'{name} 병합 완료 → shape: {df_analysis.shape}')

# 최근 90일, 180일 구매 횟수 및 최근 90일 구매한 유니크 상품 수 결측치 처리
df_analysis['frequency_last_90d']  = df_analysis['frequency_last_90d'].fillna(0).astype(int)
df_analysis['frequency_last_180d'] = df_analysis['frequency_last_180d'].fillna(0).astype(int)
df_analysis['unique_articles_last_90d'] = df_analysis['unique_articles_last_90d'].fillna(0).astype(int)

print(f'\n최종 데이터 크기: {df_analysis.shape}')
print(f'컬럼 목록: {list(df_analysis.columns)}')

RFM 병합 완료 → shape: (1143652, 6)
Recency 병합 완료 → shape: (1143652, 7)
활동성 90일 병합 완료 → shape: (1143652, 8)
활동성 180일 병합 완료 → shape: (1143652, 9)
구매 간격 병합 완료 → shape: (1143652, 11)
다양성 병합 완료 → shape: (1143652, 13)
취향 병합 완료 → shape: (1143652, 15)
[v2] 구매 트렌드 병합 완료 → shape: (1143652, 17)
[v2] 다양성 변화 병합 완료 → shape: (1143652, 19)
[v2] 구매 리듬 병합 완료 → shape: (1143652, 21)
[NEW] 시퀀스 간격 t1~t5 병합 완료 → shape: (1143652, 26)
고객 메타 병합 완료 → shape: (1143652, 31)

최종 데이터 크기: (1143652, 31)


### 8. 데이터 누수 검증
- 누수 검증 원칙
    - 누수 기준 : cut_off 이후 데이터가 피처에 포함되었는가
    - 상관관계가 높은 피처 ≠ 누수 (이탈 신호를 담은 좋은 피처일 뿐)
    - AUC > 0.95 이상이면 누수 의심 → 개별 피처 상관관계 재확인 필요

In [ ]:
# 중복 고객 확인
n_unique = df_analysis['customer_id'].nunique()
n_total  = len(df_analysis)
assert n_unique == n_total, '중복 고객 있음'
print(f'중복 없음 (총 {n_total:,}명)')

# 시퀀스 피처 churn 상관관계 확인
seq_features = [f'interval_t{i+1}' for i in range(N)]
print('\n시퀀스 피처 churn 상관관계 (절대값 0.95 이상이면 누수 의심)')

for col in seq_features:
    corr = df_analysis[col].corr(df_analysis['churn'])
    flag = '누수 의심' if abs(corr) > 0.95 else '=> 정상'
    print(f'  {col:<15}: {corr:.4f}  {flag}')


중복 없음 (총 1,143,652명)

[NEW] 시퀀스 피처 churn 상관관계 (절대값 0.95 이상이면 누수 의심)
  interval_t1    : -0.0281  ✅ 정상
  interval_t2    : -0.0234  ✅ 정상
  interval_t3    : -0.0242  ✅ 정상
  interval_t4    : -0.0211  ✅ 정상
  interval_t5    : -0.0174  ✅ 정상


In [ ]:
df_analysis.head()

### 9. 데이터 저장

In [ ]:
file_name = '../data/labeled/hnm_churn_dataset_v3.csv'
df_analysis.to_csv(file_name, index=False)

print(f'저장 완료: {file_name}')
print(f'최종 데이터 크기: {df_analysis.shape}')
print(f'이탈률: {df_analysis["churn"].mean()*100:.2f}%')
print(f'컬럼 수: 기존(26개) → 최종({df_analysis.shape[1]}개) (+{df_analysis.shape[1]-26}개: interval_t1~t5)')

### 10. 최종 데이터셋 정의서
- 모든 피처는 데이터 누수(Data Leakage)를 방지하기 위해 Cut-off 기준일(2020-03-03) 이전의 과거 531일 데이터만을 활용하여 생성

| 컬럼 그룹 | 컬럼명 | 설명 |
| :--- | :--- | :--- |
| 타겟 라벨 | churn | 이탈 여부 (Target Variable)<br>• 1: Cut-off 이후 203일간 구매 기록 없음 (이탈)<br>• 0: Cut-off 이후 203일간 최소 1회 이상 구매 (유지) |
| 고객 식별자 | customer_id | 고객 식별 ID |
| RFM 피처 | frequency | 구매 빈도: 과거 구간 동안의 총 구매 상품(아이템) 건수 |
| | price_mean | 평균 구매 금액: 유저가 구매한 상품 가격의 평균값 |
| | price_sum | 총 구매 금액: 과거 구간 동안 지출한 총 금액 |
| | price_max | 최고 구매 금액: 유저가 구매한 상품 중 가장 비싼 상품의 가격 |
| | recency | 최근성: Cut-off 기준일로부터 유저의 마지막 구매일까지 흐른 일수 |
| 활동성 피처 | frequency_last_90d | 최근 90일간의 구매 상품 건수 |
| | frequency_last_180d | 최근 180일간의 구매 상품 건수 |
| 구매 주기 | avg_purchase_interval | 평균 구매 간격: 유저의 구매 사이사이 평균 공백기(일수) |
| | std_purchase_interval | 구매 간격 편차: 구매 주기 일수의 표준편차 |
| 구매 다양성 | unique_articles | 과거 동안 구매한 서로 다른 고유 상품(Article)의 개수 |
| | unique_product_groups | 과거 동안 구매한 서로 다른 고유 카테고리(그룹)의 개수 |
| 고객 취향 | index_group_name | 가장 많이 구매한 인덱스 그룹 이름 (예: Ladieswear 등) |
| | product_group_name | 가장 많이 구매한 상품 카테고리 이름 (예: Shoes 등) |
| 고객 메타 | age | 고객의 연령(나이) |
| | FN | 패션 뉴스(Fashion News) 알림 수신 여부 (1 또는 NaN) |
| | Active | 고객의 서비스 활성화 상태 (1 또는 NaN) |
| | club_member_status | 클럽 회원 자격 상태 (예: Active, Pre-create 등) |
| | fashion_news_frequency | 패션 뉴스 수신 빈도 (예: Regular, Monthly 등) |
| 구매 트렌드 | freq_decay_rate | 구매 감소율: 최근 90일 구매 횟수 / 최근 180일 구매 횟수<br>• 1.0: 변화 없음 / 0.5: 후반 구매가 절반으로 감소 |
| | recent_price_ratio | 지출 변화율: 최근 90일 평균 구매액 / 전체 평균 구매액<br>• 1.0 미만: 최근 지출이 줄어드는 중 |
| 다양성 변화 | unique_articles_last_90d | 최근 90일간 구매한 고유 상품 수 |
| | diversity_decay_rate | 다양성 감소율: 최근 90일 고유 상품 수 / 전체 고유 상품 수<br>• 낮을수록 쇼핑 스펙트럼 축소 |
| 구매 리듬 | interval_acceleration | 구매 리듬 가속도: 마지막 구매 간격 / 평균 구매 간격<br>• 1.0 초과: 최근 들어 구매가 뜸해진 것 |
| 시퀀스 간격 | interval_t2 | 두 번째 최근 구매 간격 (결측 10.8%) |
| | interval_t3 | 세 번째 최근 구매 간격 (결측 18.9%) |
| | interval_t4 | 네 번째 최근 구매 간격 (결측 25.9%) |
| | interval_t5 | 가장 과거 구매 간격 (결측 31.4%) |